In [ ]:
%pip install osmium pandas

In [5]:
%pip install pyrosm

  Using cached pyrosm-0.6.2.tar.gz (2.5 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × installing build dependencies for pyrosm did not run successfully.
  │ exit code: 1
  ╰─> [68 lines of output]
        Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
        Using cached wheel-0.47.0-py3-none-any.whl.metadata (2.3 kB)
        Using cached cython-3.2.4-cp312-cp312-win_amd64.whl.metadata (7.7 kB)
        Using cached cykhash-2.0.0.tar.gz (43 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status 'done'
        Using cached pyrobuf-0.9.3.tar.gz (258 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build whee

In [9]:
import osmium
import csv

class BuildingHandler(osmium.SimpleHandler):

    def __init__(self, writer):
        super().__init__()
        self.writer = writer
        self.count = 0

    def way(self, w):

        # Check if object is a building
        if 'building' in w.tags:

            try:
                coords = []

                for node in w.nodes:
                    coords.append((node.lat, node.lon))

                # Skip empty geometry
                if len(coords) == 0:
                    return

                # Approximate centroid
                avg_lat = sum(c[0] for c in coords) / len(coords)
                avg_lon = sum(c[1] for c in coords) / len(coords)

                # Write to CSV
                self.writer.writerow([
                    w.id,
                    avg_lat,
                    avg_lon
                ])

                self.count += 1

                # Progress log
                if self.count % 10000 == 0:
                    print(f"{self.count} buildings extracted...")

            except:
                pass


# PBF file path
pbf_file = "india-260512.osm.pbf"

# Output CSV
output_file = "building_coordinates.csv"

with open(output_file, "w", newline="", encoding="utf-8") as f:

    writer = csv.writer(f)

    # Header
    writer.writerow(["building_id", "latitude", "longitude"])

    # Run handler
    handler = BuildingHandler(writer)
    handler.apply_file(pbf_file, locations=True)

print("Building extraction completed!")

10000 buildings extracted...
20000 buildings extracted...
30000 buildings extracted...
40000 buildings extracted...
50000 buildings extracted...
60000 buildings extracted...
70000 buildings extracted...
80000 buildings extracted...
90000 buildings extracted...
100000 buildings extracted...
110000 buildings extracted...
120000 buildings extracted...
130000 buildings extracted...
140000 buildings extracted...
150000 buildings extracted...
160000 buildings extracted...
170000 buildings extracted...
180000 buildings extracted...
190000 buildings extracted...
200000 buildings extracted...
210000 buildings extracted...
220000 buildings extracted...
230000 buildings extracted...
240000 buildings extracted...
250000 buildings extracted...
260000 buildings extracted...
270000 buildings extracted...
280000 buildings extracted...
290000 buildings extracted...
300000 buildings extracted...
310000 buildings extracted...
320000 buildings extracted...
330000 buildings extracted...
340000 buildings ex

In [12]:
import pandas as pd

# Read file
df = pd.read_csv("building_coordinates.csv")

print(df.head())

# Total rows and columns
print(df.shape)

# Column names
print(df.columns)

# Data types
print(df.dtypes)

   building_id   latitude  longitude
0      6297884  30.858947  75.860691
1      6361801  30.858105  75.860908
2      6361851  30.858126  75.861425
3      7891819  28.541961  77.167747
4     22825785  18.918761  72.847694
(16210515, 3)
Index(['building_id', 'latitude', 'longitude'], dtype='object')
building_id      int64
latitude       float64
longitude      float64
dtype: object


In [17]:
%pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import joblib

# Load dataset
df = pd.read_csv("building_coordinates.csv")

# Remove invalid rows
df = df.dropna(subset=["latitude", "longitude"])

# Convert to radians
coords = np.radians(
    df[["latitude", "longitude"]].values
)

# Build BallTree using Haversine distance
tree = BallTree(coords, metric='haversine')

# Save everything
joblib.dump(tree, "spatial_tree.pkl")
joblib.dump(df, "coordinates_data.pkl")

print("Spatial index created successfully!")

Spatial index created successfully!


In [19]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import joblib

# Load saved tree and data
tree = joblib.load("spatial_tree.pkl")
df = joblib.load("coordinates_data.pkl")

# Earth radius in meters
EARTH_RADIUS = 6371000

def get_nearby_buildings(lat, lon, radius_meters=500):

    # Convert query point to radians
    point = np.radians([[lat, lon]])

    # Convert radius to radians
    radius_radians = radius_meters / EARTH_RADIUS

    # Query tree
    indices = tree.query_radius(
        point,
        r=radius_radians
    )[0]

    # Return matching rows
    return df.iloc[indices]

# Example query
results = get_nearby_buildings(
    lat=19.0760,
    lon=72.8777,
    radius_meters=1000
)

print(results.head())
print(f"Buildings found: {len(results)}")

         building_id   latitude  longitude
2403903    358330930  19.070107  72.870952
1213054    353159534  19.070481  72.871379
1217804    353181885  19.069794  72.871835
1217793    353181871  19.070390  72.871862
1271581    353988582  19.068877  72.874134
Buildings found: 732


In [21]:
print( get_nearby_buildings(
    lat=19.0760,
    lon=72.8777,
    radius_meters=500
))

          building_id   latitude  longitude
1217637     353181465  19.074403  72.873395
13292025   1242439516  19.074777  72.873443
1217626     353181451  19.074008  72.873620
1217647     353181479  19.073613  72.873834
2748731     359780137  19.072664  72.876111
...               ...        ...        ...
1215493     353174191  19.078714  72.881264
1217650     353181482  19.078304  72.881266
1215562     353174272  19.077710  72.881703
1215534     353174239  19.077142  72.882052
1215518     353174221  19.077383  72.882171

[139 rows x 3 columns]


In [22]:
print( get_nearby_buildings(
    lat=19.0760,
    lon=72.8777,
    radius_meters=50
))

          building_id   latitude  longitude
1217623     353181448  19.076214  72.877981
10513611   1075183332  19.076164  72.877820
10498946   1073716790  19.075988  72.877945


In [23]:
print( get_nearby_buildings(
    lat=19.0760,
    lon=72.8777,
    radius_meters=100
))

          building_id   latitude  longitude
1217603     353181428  19.076501  72.878215
1217623     353181448  19.076214  72.877981
10513611   1075183332  19.076164  72.877820
10498946   1073716790  19.075988  72.877945
10500069   1073808805  19.076230  72.878224
1230092     353378242  19.075934  72.878182
1217575     353181389  19.076400  72.878441
1217618     353181443  19.076173  72.878442
1217632     353181457  19.076001  72.878451
15187978   1364174403  19.075819  72.878463
